In [1]:
import glob
import pandas as pd
import numpy as np
import os
import matplotlib.pyplot as plt

In [2]:
data = pd.read_csv('training_data/8m-resolution/US_points_raw_8m.csv')

In [3]:
data['id'] = data['id'].astype(int)
data['grid_cell_id'] = data['grid_cell_id'].astype(int)
print(data['id'].is_monotonic_increasing)

True


In [4]:
data['image_filename'] = data['id'].apply(lambda x: f'training_data/8m-resolution/US_NAIP_Semios/images/{x:07d}.tif')
data['mask_filename'] = data['id'].apply(lambda x: f'training_data/8m-resolution/US_NAIP_Semios/masks/{x:07d}.tif')

In [5]:
# Throw out image that is causing issues
data = data[data['image_filename'] != 'training_data/8m-resolution/US_NAIP_Semios/images/0003931.tif']

In [6]:
data

,system:index,grid_cell_id,id,.geo,image_filename,mask_filename
0,00000000000000000000,209,1,"{""type"":""Point"",""coordinates"":[-85.02093957527...",training_data/8m-resolution/US_NAIP_Semios/ima...,training_data/8m-resolution/US_NAIP_Semios/mas...
1,00000000000000000001,177,2,"{""type"":""Point"",""coordinates"":[-86.23359656586...",training_data/8m-resolution/US_NAIP_Semios/ima...,training_data/8m-resolution/US_NAIP_Semios/mas...
2,00000000000000000002,208,3,"{""type"":""Point"",""coordinates"":[-87.24447031557...",training_data/8m-resolution/US_NAIP_Semios/ima...,training_data/8m-resolution/US_NAIP_Semios/mas...
3,00000000000000000003,176,4,"{""type"":""Point"",""coordinates"":[-87.90077864502...",training_data/8m-resolution/US_NAIP_Semios/ima...,training_data/8m-resolution/US_NAIP_Semios/mas...
4,00000000000000000004,133,5,"{""type"":""Point"",""coordinates"":[-112.6633350792...",training_data/8m-resolution/US_NAIP_Semios/ima...,training_data/8m-resolution/US_NAIP_Semios/mas...
...,...,...,...,...,...,...
9906,000000000000000026b2,285,9907,"{""type"":""Point"",""coordinates"":[-117.5977094185...",training_data/8m-resolution/US_NAIP_Semios/ima...,training_data/8m-resolution/US_NAIP_Semios/mas...
9907,000000000000000026b3,171,9908,"{""type"":""Point"",""coordinates"":[-96.71905675641...",training_data/8m-resolution/US_NAIP_Semios/ima...,training_data/8m-resolution/US_NAIP_Semios/mas...
9908,000000000000000026b4,141,9909,"{""type"":""Point"",""coordinates"":[-97.20861700197...",training_data/8m-resolution/US_NAIP_Semios/ima...,training_data/8m-resolution/US_NAIP_Semios/mas...
9909,000000000000000026b5,316,9910,"{""type"":""Point"",""coordinates"":[-105.9947672902...",training_data/8m-resolution/US_NAIP_Semios/ima...,training_data/8m-resolution/US_NAIP_Semios/mas...


In [7]:
grid_cell_ids = np.unique(data['grid_cell_id'])
print(len(grid_cell_ids))

231


In [8]:
np.random.seed(0)
grid_cell_ids_shuffled = np.random.permutation(grid_cell_ids)
n = len(grid_cell_ids_shuffled)
grid_cell_ids_train = grid_cell_ids_shuffled[:np.rint(0.8*n).astype(int)]
grid_cell_ids_val = grid_cell_ids_shuffled[np.rint(0.8*n).astype(int):np.rint(0.9*n).astype(int)]
grid_cell_ids_test = grid_cell_ids_shuffled[np.rint(0.9*n).astype(int):]

print(len(grid_cell_ids_train))
print(len(grid_cell_ids_val))
print(len(grid_cell_ids_test))

185
23
23


In [9]:
n1 = len(grid_cell_ids_train)
n2 = len(grid_cell_ids_val)
n3 = len(grid_cell_ids_test)

data_split_list = np.concatenate((np.full(n1, 'train'), np.full(n2, 'val'), np.full(n3, 'test')))
grid_cell_ids_to_data_split = dict(zip(grid_cell_ids_shuffled, data_split_list))

In [10]:
data['data_split'] = data['grid_cell_id'].apply(lambda x: grid_cell_ids_to_data_split[x])

In [11]:
# Originally held out 500 points for calibration, but ended up not using them since they are not randomly sampled over all US croplands
# Instead we manually labeled a separate calibration set of 500 points (see calibration Jupyter notebooks)

data.loc[data.sample(n=500, random_state=0).index, 'data_split'] = 'not_used'

In [8]:
data

,system:index,grid_cell_id,id,.geo,image_filename,mask_filename,data_split
0,00000000000000000000,209,1,"{""type"":""Point"",""coordinates"":[-85.02093957527...",training_data/8m-resolution/US_NAIP_Semios/ima...,training_data/8m-resolution/US_NAIP_Semios/mas...,train
1,00000000000000000001,177,2,"{""type"":""Point"",""coordinates"":[-86.23359656586...",training_data/8m-resolution/US_NAIP_Semios/ima...,training_data/8m-resolution/US_NAIP_Semios/mas...,train
2,00000000000000000002,208,3,"{""type"":""Point"",""coordinates"":[-87.24447031557...",training_data/8m-resolution/US_NAIP_Semios/ima...,training_data/8m-resolution/US_NAIP_Semios/mas...,train
3,00000000000000000003,176,4,"{""type"":""Point"",""coordinates"":[-87.90077864502...",training_data/8m-resolution/US_NAIP_Semios/ima...,training_data/8m-resolution/US_NAIP_Semios/mas...,train
4,00000000000000000004,133,5,"{""type"":""Point"",""coordinates"":[-112.6633350792...",training_data/8m-resolution/US_NAIP_Semios/ima...,training_data/8m-resolution/US_NAIP_Semios/mas...,test
...,...,...,...,...,...,...,...
9905,000000000000000026b2,285,9907,"{""type"":""Point"",""coordinates"":[-117.5977094185...",training_data/8m-resolution/US_NAIP_Semios/ima...,training_data/8m-resolution/US_NAIP_Semios/mas...,train
9906,000000000000000026b3,171,9908,"{""type"":""Point"",""coordinates"":[-96.71905675641...",training_data/8m-resolution/US_NAIP_Semios/ima...,training_data/8m-resolution/US_NAIP_Semios/mas...,train
9907,000000000000000026b4,141,9909,"{""type"":""Point"",""coordinates"":[-97.20861700197...",training_data/8m-resolution/US_NAIP_Semios/ima...,training_data/8m-resolution/US_NAIP_Semios/mas...,train
9908,000000000000000026b5,316,9910,"{""type"":""Point"",""coordinates"":[-105.9947672902...",training_data/8m-resolution/US_NAIP_Semios/ima...,training_data/8m-resolution/US_NAIP_Semios/mas...,val


In [9]:
data['data_split'].value_counts()

data_split
train       7111
val         1469
test         830
not_used     500
Name: count, dtype: int64

In [10]:
data.to_csv('training_data/8m-resolution/US_points_final_8m.csv', index=False)